<a href="https://colab.research.google.com/github/Bersanone/ArchitteturaTransformer/blob/main/ArchitetturaTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Caricamento dati

In [ ]:
with open("tiny_shakespeare.txt", "r") as f:
  data = f.read()

In [ ]:
print(data[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


# Costruzione del dizionario

In [ ]:
dictionary = sorted(list(set(list(data))))

In [ ]:
itos = {i:c for i, c in enumerate(dictionary)}
stoi = {c:i for i, c in enumerate(dictionary)}

# Costruzione del training set

In [ ]:
CONTEXT_LEN = 128

In [ ]:
xs = []
ys = []
for i in range(CONTEXT_LEN, len(data)-1):
  x = data[i-CONTEXT_LEN:i]
  y = data[i-CONTEXT_LEN+1:i+1]
  xs.append([stoi[c] for c in x])
  ys.append([stoi[c] for c in y])

In [ ]:
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs.shape, ys.shape

(torch.Size([1115265, 128]), torch.Size([1115265, 128]))

# Split training e test set

In [ ]:
torch.random.manual_seed(42)

Mischia gli indici e poi compone il training set (primo 90% degli indici mischiati) e il test set (restante 10%)

In [ ]:
perm_idxs = torch.randperm(xs.shape[0])

In [ ]:
threshold = int(0.9*xs.shape[0]) # l'indice corripondente al 90% dei dati

xs_train = xs[perm_idxs[:threshold]]
ys_train = ys[perm_idxs[:threshold]]

xs_test = xs[perm_idxs[threshold:]]
ys_test = ys[perm_idxs[threshold:]]

# Architettura del Transformer Decoder

## Strato di embedding

In [ ]:
class TransformerEmbedding(nn.Module):
  def __init__(self, sequence_len, embedding_dim, vocab_size):
    super().__init__()
    # codifica i token (ha una lunghezza pari al vocabolario)
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    # codifica le posizioni (ha una lunghezza pari alle possibili posizioni)
    self.position_embedding_table = nn.Embedding(sequence_len, embedding_dim)

  def forward(self, input_ids):
    """ Metodo forward

    Args:
      input_ids: gli indici dei token
    """
    batch_size, sequence_len = input_ids.shape

    # genero gli embedding per i token
    token_embedding = self.token_embedding_table(input_ids)

    # genero gli embedding per le posizioni
    position_embedding = self.position_embedding_table(torch.arange(sequence_len, device=input_ids.device))

    # sommo gli embedding per i token e le posizioni
    # broadcasting in azione perché token_embedding ha
    # shape (batch_size x sequence_len x embedding_dim)
    # mentre position_embedding ha shape (sequence_len x embedding_dim)
    return token_embedding + position_embedding


## Test embedding layer

In [ ]:
embedding_layer = TransformerEmbedding(
    sequence_len=CONTEXT_LEN, # lunghezza delle x
  	embedding_dim=164, # lunghezza dell'embedding
  	vocab_size=len(dictionary) # lunghezza del dizionario
)

In [ ]:
fake_input = torch.randint(
    low=0,
    high=len(dictionary),
    size=(32, CONTEXT_LEN)
)

In [ ]:
fake_input.shape

torch.Size([32, 128])

In [ ]:
fake_output = embedding_layer(fake_input)

In [ ]:
fake_output.shape

torch.Size([32, 128, 164])

## Decoder block

In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self, embed_dim, num_heads, mlp_dim, dropout_rate=0.1):
    super().__init__()

    # attenzione
    self.att = nn.MultiheadAttention(
        embed_dim=embed_dim,
        num_heads=num_heads,
        batch_first=True
    )

    # MLP
    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim, out_features=mlp_dim),
        nn.GELU(),
        nn.Linear(in_features=mlp_dim, out_features=embed_dim)
    )

    # normalizzazione
    self.norm1 = nn.LayerNorm(normalized_shape=embed_dim) # input = batch_size x seq_len x embed_dim
    self.norm2 = nn.LayerNorm(normalized_shape=embed_dim) # input = batch_size x seq_len x embed_dim

    # dropout
    self.dropout1 = nn.Dropout(p=dropout_rate)
    self.dropout2 = nn.Dropout(p=dropout_rate)

    # attention mask
    # self.attn_mask = torch.triu(torch.ones(size=(CONTEXT_LEN, CONTEXT_LEN)), diagonal=1).to(torch.bool)
    self.register_buffer("attn_mask", torch.triu(torch.ones(size=(CONTEXT_LEN, CONTEXT_LEN)), diagonal=1).to(torch.bool))

  def forward(self, x):
    y, _ = self.att(x, x, x, is_causal=True, attn_mask=self.attn_mask) # attenzione restituisce una tupla (il secondo elemento è la matrice di attenzione)
    y = self.dropout1(y)
    y = self.norm1(x + y)

    z = self.mlp(y)
    z = self.dropout2(z)
    z = self.norm2(y + z)

    return z

## Test decoder block

In [ ]:
block = DecoderBlock(embed_dim=164, num_heads=4, mlp_dim=256)

In [ ]:
fake_imput = torch.randn(size=(32, CONTEXT_LEN, 164))

In [ ]:
fake_output = block(fake_imput)

In [ ]:
fake_output.shape

torch.Size([32, 128, 164])

## Transformer Decoder

In [ ]:
EMBEDDING_DIM = 168
N_HEADS = 8
N_LAYER = 4
DROPOUT = 0.2
MLP_DIM = 256

In [ ]:
class TransformerDecoder(nn.Module):
  def __init__(self, vocab_size, sequence_len, embedding_dim, n_layer, dropout, n_heads, mlp_dim):
    super().__init__()

    # embedding
    self.embedding = TransformerEmbedding(
        sequence_len=sequence_len,
        embedding_dim=embedding_dim,
        vocab_size=vocab_size
    )

    # sequenza dei decoder block
    self.blocks = nn.Sequential(
        *[DecoderBlock(
            embed_dim=embedding_dim,
            num_heads=n_heads,
            mlp_dim=mlp_dim,
            dropout_rate=dropout
        ) for _ in range(n_layer)]
    )

    # classificatore finale
    self.final_dense = nn.Linear(in_features=embedding_dim, out_features=vocab_size)

  def forward(self, x):

    # embedding
    x = self.embedding(x)

    # decoder blocks
    for block in self.blocks:
      x = block(x)

    # classificazione
    x = self.final_dense(x)

    return x

## Test del transformer decoder

In [ ]:
transformer = TransformerDecoder(
    vocab_size=len(dictionary),
    sequence_len=CONTEXT_LEN,
    embedding_dim=EMBEDDING_DIM,
    n_layer=N_LAYER,
    dropout=DROPOUT,
    n_heads=N_HEADS,
    mlp_dim=MLP_DIM
)

In [ ]:
fake_input = torch.randint(low=0, high=len(dictionary), size=(32, CONTEXT_LEN))

In [ ]:
fake_output = transformer(fake_input)

In [ ]:
fake_output.shape

torch.Size([32, 128, 65])

# Routine di addestramento


In [ ]:
xs.shape, ys.shape

(torch.Size([1115265, 128]), torch.Size([1115265, 128]))

In [ ]:
model = TransformerDecoder(
    vocab_size=len(dictionary),
    sequence_len=CONTEXT_LEN,
    embedding_dim=EMBEDDING_DIM,
    n_layer=N_LAYER,
    dropout=DROPOUT,
    n_heads=N_HEADS,
    mlp_dim=MLP_DIM
)

In [ ]:
def select_random_samples(x, y, n_samples):
  """ Seleziona casualmente n_samples elementi da x e y"""
  perm_idxs = torch.randperm(x.shape[0])
  return x[perm_idxs[:n_samples]], y[perm_idxs[:n_samples]]

In [ ]:
def generate_text(model, n_tokens, seed_text, device, nucleus_p=0.9):
  # allunga il seed_text con spazi iniziali per arrivare a una lunghezza di CONTEXT_LEN
  seed_text = " " * (CONTEXT_LEN - len(seed_text)) + seed_text

  token_list = [stoi[c] for c in seed_text]
  model.to(device)
  model.eval()

  for _ in range(n_tokens):
    x = torch.tensor(token_list[-CONTEXT_LEN:]).unsqueeze(0).to(device)
    y = model(x) # 1 x x.shape[1] x dictionary_len
    logits = y[0, -1, :] # dictionary_len


    # top p selection
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_idxs = torch.sort(probs, descending=True)

    cumsum = 0
    for i, p in enumerate(sorted_probs):
      cumsum += p
      if cumsum >= nucleus_p:
        break

    sorted_idxs = sorted_idxs[:i+1]
    sorted_probs = sorted_probs[:i+1]

    idx = torch.multinomial(sorted_probs, num_samples=1)
    token = sorted_idxs[idx]
    token_list.append(token.item())

  return "".join([itos[t] for t in token_list])

In [ ]:
MAX_EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
N_SAMPLES = 70_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model.to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(MAX_EPOCHS):
  # generazione del testo

  text = generate_text(
      model=model,
      n_tokens=1000,
      seed_text="To be or not to be",
      device="cuda",
      nucleus_p=0.9
  )

  print(f"Epoch {epoch+1}: {text}")

  losses = []
  x_train_epoch, y_train_epoch = select_random_samples(xs_train, ys_train, N_SAMPLES)
  for idx in range(0, N_SAMPLES, BATCH_SIZE):
    x_batch = x_train_epoch[idx:idx+BATCH_SIZE].to(DEVICE)
    y_batch = y_train_epoch[idx:idx+BATCH_SIZE].to(DEVICE)
    y_pred = model(x_batch)
    loss = F.cross_entropy(y_pred.reshape(-1, y_pred.shape[-1]), y_batch.reshape(-1))
    losses.append(loss.item())

    # calcolo del gradiente
    loss.backward()

    # step di discesa
    optimizer.step()
    optimizer.zero_grad()

  print(f"Epoch {epoch+1}: loss = {sum(losses)/len(losses)}")

  # salva il modello
  torch.save(model.state_dict(), f"checkpoints/model_{epoch:03d}.pt")

Epoch 1:                                                                                                               To be or not to bene
Of this not melvie yea, fire life sings,
I do ling, and they so dear mostatin'st;
Is warl an our do bed; by mons bettle within'd
Contishon'd an wall a the reting pright intell.
In him in the richiefit my like fetch from unage.

BIONDES:
I thou best name ear fearst thee?

MARCIUS:
By then the hear arget not from without wither.

BUCENTIO:
You servents poor them baster this followary,
To complent him let all hone than I think
Our a precy for eyes may by night, with deven of yours
what I hear of hear confiation,
Serpate of eals, let and the foollow you
they was when all is the bearl.
Now you had lefore and thy sorrow,
A father without and and a stay that must.
I him will there traises one one?
Should that concely alone intent.

POOLIXENES:
O, be incentent.

QUEEN ELABETELA:
I prove youtht fellown his been her father:
You trune speak them forgest, with

KeyboardInterrupt: 

In [ ]:
generate_text(
    model=model,
    n_tokens=100,
    seed_text="To be or not to be",
    device="cuda",
    nucleus_p=0.9
)

"                                                                                                              To be or not to be-TVzdTTizP zKnOL:qmbdkhPPip'V'rz\nGsl!GbSIfFKETL,TfEv\n:ioI:sniwshe3pkxI:ap!gQS-mz&b dfS3dHvZVSIwwzBL$"

# Caricamento checkpoint

In [ ]:
model = TransformerDecoder(
    vocab_size=len(dictionary),
    sequence_len=CONTEXT_LEN,
    embedding_dim=EMBEDDING_DIM,
    n_layer=N_LAYER,
    dropout=DROPOUT,
    n_heads=N_HEADS,
    mlp_dim=MLP_DIM
)

In [ ]:
state_dict = torch.load("checkpoints/model_001.pt")

In [ ]:
model.load_state_dict(state_dict)

<All keys matched successfully>